# Problem 4: Automation
Create a GitHub Actions workflow to run your script every Saturday morning. The script should be called faang.yml in a .github/workflows/ folder in the root of your repository. In your notebook, explain each of the individual lines in your workflow.


The hidden folder <font color="crimson">.github/workflows/</font> contains a single YAML file that define the automated workflows.
The YAML file contains specifically:
* the Workflow name
* the CRON-based scheduled trigger
* Job (what to run)
* Steps (how to run)

***
# Workflow Automated Schedule
A Github automated Workflow was defined within the YAML file [faang.yaml](./.github/workflows/faang.yaml) to trigger the python script [faang.py](./faang.py) each day @13:00  
The workflow is called **"Download and Plot FAANG Data"**, and is found in my Github repository (https://github.com/ngn73/8645-computer-infrastructure) Actions Tab.   
  
<img src="./images/Actions.png" width="1000">  
  

### Setting Schedule 
Within the YAML file the schedule is defined with a Cron-based configuration   
https://www.advsyscon.com/blog/python-job-scheduling/   
https://docs.github.com/en/actions/reference/workflows-and-actions/events-that-trigger-workflows#schedule   
  
It is setup to run @13:00 on a daily basis.  
```python
on:
  schedule:
    - cron: "0 13 * * *"   # daily @13:00 UTC 
  workflow_dispatch:        # manual start button
```    

* It should be noted that this workflow usually does not trigger until about 13:25 (or later) due to the long queues in the GitHub Cloud



***
# Workflow Subprocesses   

There are several nested subprocesses that run within the "Download and Plot FAANG Data" workflow
1. Check out your repository
2. Set up Python
3. Install dependencies (with requirements.txt)
4. Run script
5. Upload logs, CSVs and images as artifacts

### 1. Check out your repository   
Before ruuning the Pyhon script, we need to inially checkout the repository with the following

```python
- name: Check out repository
        uses: actions/checkout@v4
```

### 2. Set up Python  
We need to next make a specific Python version available on the runner.  
The Python Version used in my codespace was checked with :  
```python
python --version
```   
  
This (on my codespace) is "Python 3.12.1" and it is configured in yaml file as follows:


```python
- name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.12'   # my version
```  

### 3. Install dependencies (with requirements.txt)   
The project dependecies defined in the [requirements.txt](requirements.txt) file is used load all required dependant python Libraries  


```python
      - name: Install dependencies
        run: |
          if [ -f requirements.txt ]; then
            pip install -r requirements.txt
          fi
```   
  

### 4. Run Python Script
With python versioning, and dependancies sorted, we can now simply execute the main script [faang.py](faang.py)   

   ```python
      - name: Run Python Script
        run: python faang.py
   ```  

### 5. Upload logs, CSVs and images as artifacts
Within the yaml file I also setup Github Workflow **Artifiacts** (detailed below)

***
## GitHub Actions Artifacts

In researching Github actions, I encountered the use of "Artifacts"  
According to a [github document](https://docs.github.com/en/actions/concepts/workflows-and-actions/workflow-artifacts), An Artifact is:
> ... a file or collection of files produced during a workflow run. Artifacts allow you to persist data after a job has completed, and share that data with another job in the same workflow. For example, you can use artifacts to save your build and test output after a workflow run has ended.  

The "faang" project outputs several files when it is executed.
* CSV Files
* PNG Image Files
* Log files  
  

Github Action Artifacts are ideal for storing the output dumps of Github actions (CSV,Images, Logs, etc.)    
It also reduces the clutter of multiple files in Repo    
https://chatgpt.com/share/6939defd-1500-8001-bdbb-3eb14b56a475
https://docs.github.com/en/actions/concepts/workflows-and-actions/workflow-artifacts  
https://earthly.dev/blog/github-action-artifacts/    

The following section was used to define the files to be stored in the workflow artifacts

```python
# Upload logs, CSVs and images as artifacts
- name: Upload output artifacts
uses: actions/upload-artifact@v4
with:
    name: script-output
    path: |
    data/archive/*.csv
    logs/*.log
    data/plots/*.png
    if-no-files-found: warn 
``` 



## Commit Archive to Repo ... or ...  Save to Workflow Artifacts ?   
  
After looking into Workflow Artifacts, I had questioned the need to be archiving files and commiting these Archives to my GitHub Repo.  
With the time spent writing the code, I was reluctant to completly scrap my code that uses 'archive' and 'staging' folders since discovering Artifacts.      
Futhermore I had reached out to Ian McLoughlin (Module Lecturer) who responded:  
> "How would that work for someone who clones your repository to their own machine?
> Would they still see your data and plots?"
  
I decided to **keep both approaches** with :  
  * Archives of Logs, CSV, and PNG files stored in GitHub Repo  
  * CSV and PNG files saved to Artifact Zip file (downloadable from repo)  

So the following YAML code was used to both ...   
1. Automatically Add/Commit/Push archived files to Repo
2. Save output files to a (zipped) Archive file   
  
```python
        - name: Commit generated outputs to repo
        run: |
          git config user.name "github-actions"
          git config user.email "actions@github.com"

          git add logs/*.log data/*.csv data/archive/*.cvs data/plots/*.png || true

          git status
          git commit -m "Add generated outputs [skip ci]" || echo "Nothing to commit"
          git push

        - name: Upload output artifacts
        uses: actions/upload-artifact@v4
        with:
          name: script-output
          path: |
            logs/*.log
            data/plots/*.png
            data/archive/*.csv
            data/*.csv
          if-no-files-found: warn
```